# iSAGE — Iterative Sparse Annotation Guided by Expert

A 3-cell workflow:

1. **Setup** — imports.
2. **Configure session** — pick dataset, training config, session, iteration. Click *Setup / Load*. Need a new dataset? Expand *Create new dataset…* at the bottom.
3. **Loop**: annotate → train → repeat. Each iteration's state lives under `Sessions/<name>/iteration_N/`.


In [ ]:
# Cell 1 — Setup

import sys
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imageio
from tqdm import tqdm
import ipywidgets as W
from IPython.display import display, clear_output

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.config_loader import load_dataset_config, load_training_config
from src.session.manager import get_or_create_session
from src.session.session_view import SessionView
from src.session.mask_utils import batch_json_to_masks
from src.session.simple_mask_converter import convert_mask_to_json
from src.training.setup import setup_training
from src.training.workflow import run_training_iteration
from src.annotation.launcher import run_annotation_workflow

import torch
print(f"PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

import segmentation_models_pytorch as smp
print(f"smp from: {smp.__file__}")

%matplotlib inline


## Cell 2 — Configure session

The session status panel below the **Session** field updates as you pick a name —
it shows existence, iteration count, completeness per iteration, and val mIoU
history when available.

Click *Setup / Load* to materialize the model, losses, and session on disk.
The **Use iter:** dropdown defaults to *latest* but can target any past iteration.

To register a new dataset, expand the *Create new dataset…* section at the bottom.
You only need to do this once per dataset.


In [ ]:
# Cell 2 — Session manager widget

_datasets_dir = Path('configs/datasets')
_trainings_dir = Path('configs/training')
_sessions_dir = Path('Sessions')

def _scan_dataset_options():
    return sorted(f.name for f in _datasets_dir.glob('*.yaml')) if _datasets_dir.exists() else []

def _scan_training_options():
    return sorted(f.name for f in _trainings_dir.glob('*.yaml')) if _trainings_dir.exists() else []

def _scan_session_options():
    return sorted(d.name for d in _sessions_dir.iterdir() if d.is_dir()) if _sessions_dir.exists() else []

_dataset_options  = _scan_dataset_options()
_training_options = _scan_training_options()
_existing_sessions = _scan_session_options()

_default_training = 'unet_efficientnet_b7.yaml' if 'unet_efficientnet_b7.yaml' in _training_options else (_training_options[0] if _training_options else None)
_default_dataset  = _dataset_options[0] if _dataset_options else None
_default_session  = _existing_sessions[0] if _existing_sessions else 'my_run'

_wlayout = W.Layout(width='620px')
_wstyle = {'description_width': '110px'}

_dataset_w   = W.Dropdown(options=_dataset_options, value=_default_dataset, description='Dataset:', layout=_wlayout, style=_wstyle)
_training_w  = W.Dropdown(options=_training_options, value=_default_training, description='Training:', layout=_wlayout, style=_wstyle)
_session_w   = W.Combobox(options=_existing_sessions, value=_default_session, placeholder='session name', description='Session:', layout=_wlayout, style=_wstyle, ensure_option=False)
_status_html = W.HTML(value='')
_iteration_w = W.Dropdown(options=[('latest', 'latest')], value='latest', description='Use iter:', layout=_wlayout, style=_wstyle)
_btn         = W.Button(description='Setup / Load', button_style='primary', icon='play', layout=W.Layout(width='220px'))
_out         = W.Output()

def _refresh_session_status(change=None):
    view = SessionView(_sessions_dir / _session_w.value)
    _status_html.value = view.summary_html()
    if view.exists and view.iterations:
        opts = [('latest', 'latest')] + [(f'iteration_{s.number}', s.number) for s in view.iterations]
    else:
        opts = [('latest', 'latest')]
    _iteration_w.options = opts
    _iteration_w.value = 'latest'

_session_w.observe(_refresh_session_status, names='value')
_refresh_session_status()

def _setup_and_load(_):
    global dataset_config, training_config, model, device, train_loss, val_loss, metrics, optimizer
    global SESSION_PATH, CURRENT_ITERATION
    with _out:
        clear_output()
        try:
            dataset_config  = load_dataset_config(f'configs/datasets/{_dataset_w.value}')
            training_config = load_training_config(f'configs/training/{_training_w.value}')
            print(f"Dataset:  {dataset_config['name']} - {dataset_config['classes']['num_classes']} classes ({', '.join(dataset_config['classes']['names'])})")
            print(f"Training: {training_config['name']} - {training_config['model']['architecture']} + {training_config['model']['encoder']}")
            print()
            model, device, train_loss, val_loss, metrics, optimizer = setup_training(
                dataset_config=dataset_config, training_config=training_config,
            )
            print()
            SESSION_PATH = Path(f'Sessions/{_session_w.value}')
            session_info = get_or_create_session(
                session_path=SESSION_PATH, dataset_config=dataset_config,
            )
            CURRENT_ITERATION = _iteration_w.value
            print(f"\nReady. Will operate on iteration = {CURRENT_ITERATION}")
            _refresh_session_status()
        except Exception as e:
            print(f"FAILED: {type(e).__name__}: {e}")
            import traceback; traceback.print_exc()

_btn.on_click(_setup_and_load)

# --- New Dataset accordion (collapsed by default) ---
_new_name    = W.Text(placeholder='my_dataset', description='Name:', layout=W.Layout(width='540px'), style={'description_width': '140px'})
_new_train_p = W.Text(placeholder='/path/to/images', description='Train images dir:', layout=W.Layout(width='540px'), style={'description_width': '140px'})
_new_val_p   = W.Text(placeholder='(optional)', description='Val images dir:', layout=W.Layout(width='540px'), style={'description_width': '140px'})
_new_valm_p  = W.Text(placeholder='(optional)', description='Val masks dir:', layout=W.Layout(width='540px'), style={'description_width': '140px'})
_new_classes = W.Textarea(placeholder='road\nbuilding\nvegetation', description='Class names:', layout=W.Layout(width='540px', height='90px'), style={'description_width': '140px'})
_new_btn     = W.Button(description='Create dataset YAML', button_style='success', icon='plus', layout=W.Layout(width='260px'))
_new_status  = W.HTML(value='')

def _create_dataset(_):
    name = _new_name.value.strip()
    train_p = _new_train_p.value.strip()
    val_p = _new_val_p.value.strip() or None
    valm_p = _new_valm_p.value.strip() or None
    class_list = [c.strip() for c in _new_classes.value.split('\n') if c.strip()]

    if not name or not train_p or not class_list:
        _new_status.value = '<span style="color:#c00">Name, train images path, and class names are required.</span>'
        return
    train_dir = Path(train_p)
    if not train_dir.exists():
        _new_status.value = f'<span style="color:#c00">Train images path does not exist: {train_dir}</span>'
        return

    sample_imgs = (
        sorted(train_dir.glob('*.png')) + sorted(train_dir.glob('*.tif'))
        + sorted(train_dir.glob('*.tiff')) + sorted(train_dir.glob('*.jpg'))
    )
    if not sample_imgs:
        _new_status.value = f'<span style="color:#c00">No images found in {train_dir}</span>'
        return

    with Image.open(sample_imgs[0]) as img:
        width, height = img.size
        channels = len(img.getbands())

    num_classes = len(class_list)
    ignore_index = num_classes
    cmap = plt.get_cmap('tab10' if num_classes <= 10 else 'tab20')
    colors = [[int(255 * c) for c in cmap(i)[:3]] for i in range(num_classes)]

    yaml_obj = {
        'name': name.upper(),
        'paths': {
            'train_images': str(train_dir),
            'train_dense_masks': None,
            'train_sparse_masks': str(Path('Sessions') / name / 'sparse_masks'),
            'val_images': val_p,
            'val_masks': valm_p,
            'test_images': None,
            'test_masks': None,
        },
        'classes': {
            'num_classes': num_classes,
            'ignore_index': ignore_index,
            'names': class_list,
            'colors': colors,
        },
        'image': {'width': width, 'height': height, 'channels': channels},
    }
    import yaml as _yaml
    yaml_path = _datasets_dir / f'{name}.yaml'
    if yaml_path.exists():
        _new_status.value = f'<span style="color:#c00">{yaml_path.name} already exists - pick another name.</span>'
        return
    with open(yaml_path, 'w') as f:
        _yaml.safe_dump(yaml_obj, f, default_flow_style=False, sort_keys=False)

    val_note = '' if val_p else ' (no validation — that is fine)'
    _new_status.value = (
        f'<span style="color:#2a7"><b>Created {yaml_path.name}</b> - '
        f'{len(sample_imgs)} images, {width}x{height}x{channels}{val_note}</span>'
    )
    _dataset_w.options = _scan_dataset_options()
    _dataset_w.value = yaml_path.name

_new_btn.on_click(_create_dataset)

_new_form = W.VBox([_new_name, _new_train_p, _new_val_p, _new_valm_p, _new_classes, _new_btn, _new_status])
_accordion = W.Accordion(children=[_new_form])
_accordion.set_title(0, 'Create new dataset...')
_accordion.selected_index = None

display(W.VBox([_dataset_w, _training_w, _session_w, _status_html, _iteration_w, _btn, _out, _accordion]))


## Cell 3 — Annotate

Launches the PyQt5 annotation widget on `CURRENT_ITERATION` (from the widget above).
Left-click adds a point for the current class; right-click removes a nearby point.
Close the window to advance.


In [ ]:
# Cell 3 — Annotate

result = run_annotation_workflow(
    session_path=SESSION_PATH,
    dataset_config=dataset_config,
    iteration=CURRENT_ITERATION,
    launch_tool=True,
)


## Cell 4 — Train

Trains the model on `CURRENT_ITERATION`'s annotations using EWDL, generates
predictions, and advances to the next iteration. Plots mIoU progression below
when validation data exists.


In [ ]:
# Cell 4 — Train

result = run_training_iteration(
    session_path=SESSION_PATH,
    dataset_config=dataset_config,
    training_config=training_config,
    model=model,
    device=device,
    train_loss=train_loss,
    val_loss=val_loss,
    metrics=metrics,
    optimizer=optimizer,
    iteration=CURRENT_ITERATION,
    visualize=False,
)

# Plot mIoU history (only meaningful if val data exists)
_csv = SESSION_PATH / 'metrics_history.csv'
if _csv.exists() and _csv.stat().st_size > 0:
    _h = pd.read_csv(_csv)
    if len(_h) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(_h['iteration'], _h['miou'], marker='o', linewidth=2, color='#1f77b4', label='val mIoU')
        if 'pixel_accuracy' in _h.columns:
            ax.plot(_h['iteration'], _h['pixel_accuracy'], marker='s', linewidth=1.5, color='#7f7f7f', alpha=0.6, label='pixel acc')
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Metric')
        ax.set_title(f'{SESSION_PATH.name} - progression')
        ax.grid(alpha=0.3); ax.legend(); ax.set_ylim(0, 1)
        plt.tight_layout(); plt.show()
else:
    print('(No metrics_history.csv yet - either first training or no validation set configured.)')
